In [ ]:
from pathlib import Path
import os
from os import PathLike
from typing import Optional, Sequence, Union
from fastcore.basics import patch

from trouver.obsidian.vault import NoteDoesNotExistError, VaultNote, note_name_from_path, all_paths_to_notes_in_vault, NoteNotFoundInCacheError, NoteNotUniqueError, NotePathIsNotIdentifiedError
from trouver.obsidian.links import ObsidianLink, LinkType, replace_links_in_text
from trouver.obsidian.vault import path_to_obs_id

In [ ]:
import shutil
import tempfile
from unittest import mock

from fastcore.test import *
from nbdev.showdoc import show_doc

from trouver.helper.tests import _test_directory

## VaultNote class
Just as how paths in Python can be dealt either via strings of paths or via `pathlib.Path` objects, It is useful to have a class to encapsulate together the name of a note, and its path/Obsidian vault identifier.

In [ ]:
#| export obsidian.vault
# TODO: test hidden methods
class VaultNote:
    r"""Represents a note in an Obsidian vault, without regards to the contents.
    
    The note does not have to exist, except in circumstances stating 
    otherwise.

    TODO go through the methods of this class to see which methods assume that
    the note exists and which do not.

    TODO finish the sentence below.
    A `VaultNote` can be specified by either the `rel_path` or the `name` argument
    in its constructor. If `name` is specified, then the 
    
    TODO implement subdirectory hint
    
    **Attributes**

    - vault - Path
        - The (relative or absolute) path of the Obsidian vault
        that the note is located in.
    - name - str
        - The name of the note in the vault.
    - rel_path - str
        - The note's path relative to the vault. If 
    - cache - dict[str, dict[str, list[str]]], class attribute
        - The keys are string, which are paths to vaults. The
        corresponding values are dict whose keys are string, which are
        names in the vault of the (unique) note of that name, and whose
        values are list of string, which are paths to the note relative to the
        vault. The cache is not automatically updated when notes are
        moved, unless the `.move_to` method or its derivatives are invoked.
    
    **Parameters**

    - vault - PathLike
    - rel_path - PathLike
    - name - str
        - The name of the note in the vault. Defaults to the empty str.
            - If `None`, then the `rel_path` parameter should be used 
            to determine `self.name` instead. 
            - If not `None`, then the note must uniquely exist in the
            vault.
    - `subdirectory` - Union[PathLike, None]
    - `hints` - list[PathLike]

    **Raises**
    
    - ValueError
        - if `rel_path` and `name` are both `None`.
    """
    
    cache = {}
    
    def __repr__(self):
        # if self.rel_path and self.
        if self.rel_path_identified():
            if self.name:
                return f"VaultNote(vault={repr(self.vault)}, name={self.name}, rel_path={repr(self.rel_path)})"
            else:
                return f"VaultNote(vault={repr(self.vault)}, rel_path={repr(self.rel_path)})"
        else:
            return f"VaultNote(vault={repr(self.vault)}, name={self.name})"

    @classmethod
    def _check_if_cache_needs_to_update(
            cls, vault: PathLike, name: str) -> bool:
        r"""Returns `True` if the cache needs to update by virtue of not finding
        notes of the specified `name` in the `vault`.
        """
        return not bool(cls._get_from_cache(vault, name))

In [ ]:
#| export obsidian.vault
@patch(cls_method=True)
def _get_from_cache(
        cls: VaultNote,
        vault: PathLike,
        name: str,
        ) -> Union[list[str], None]:
    r"""Return the cache's list of notes of the specified name in the
    specified vault.

    If no such list exists in the cache, then return `None`.
    """
    vault = str(vault)
    if vault not in cls.cache:
        return None
    vault_dict = cls.cache[vault]
    if not name in vault_dict:
        return None
    return vault_dict[name]

## Constructing VaultNote instances

In [ ]:
#| export obsidian.vault
@patch
def __init__(
        self: VaultNote,
        vault: PathLike, # The (relative or absolute) path of the Obsidian vault that the note is located in.
        rel_path: PathLike = None, # The note's path relative to the vault. If `None`, then the `name` parameter is used to determine the note instead. Defaults to `None`.
        name: str = None, # The name of the note. If `None`, then the `rel_path` parameter is used to determine the note instead. Defaults to `None` 
        subdirectory: Union[PathLike, None] = '', # The relative path to a subdirectory in the Obsidian vault. If `None`, then denotes the root of the vault. Defaults to the empty str. 
        hints: list[PathLike] = [], # Paths, relative to `subdirectory`, to directories where the note file may be found. This is for speedup. Defaults to the empty list, in which case the vault note is searched in all of `subdirectory`.
        update_cache: Optional[bool] = True # If `True` and if `rel_path` is not specified (and hence `name` is specified), update the cache
        ):
    # TODO: consider using _check_name_exists_and_unique_in_vault_cache method.
    self.vault = Path(vault)
    if rel_path is None and name is None:
        raise ValueError(
            "In constructing a `VaultNote` object, the parameters `rel_path`"
            " and `name` parameters were expected to be given arguments, but"
            " both parameters are given `None` as arguments.")
    if rel_path is not None:
        self.rel_path = str(rel_path)
        self.name = note_name_from_path(self.rel_path)
    else:
        self.name = name
        self.rel_path = None
        self.identify_rel_path(update_cache=update_cache)